In [0]:
# This notebook copies the Genie Code skills from this repo into your personal skills folder.
# Run it once per workspace after deploying or cloning the project, before you start adapting the template with Genie Code.

In [0]:
dbutils.widgets.text("source_skills_path", "", "Source skills path (optional)")

source_skills_path_override = dbutils.widgets.get("source_skills_path").strip()

In [0]:
import os


def to_file_uri(path: str) -> str:
    return path if path.startswith("file:") else f"file:{path}"


def path_exists(path: str) -> bool:
    if os.path.isdir(path):
        return True
    try:
        dbutils.fs.ls(to_file_uri(path))
        return True
    except Exception:
        return False


def resolve_source_skills_path() -> str:
    if source_skills_path_override:
        if not path_exists(source_skills_path_override):
            raise ValueError(f"source_skills_path not found: {source_skills_path_override}")
        return source_skills_path_override

    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    candidates = []

    if "/files/notebooks/" in notebook_path:
        files_root = notebook_path.split("/files/notebooks/")[0] + "/files"
        candidates.append(os.path.join(files_root, ".assistant", "skills"))

    if "/notebooks/" in notebook_path:
        repo_root = notebook_path.split("/notebooks/")[0]
        candidates.append(os.path.join(repo_root, ".assistant", "skills"))

    cwd = os.getcwd()
    candidates.append(os.path.join(cwd, ".assistant", "skills"))
    candidates.append(os.path.join(os.path.dirname(cwd), ".assistant", "skills"))

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        if path_exists(path):
            return path

    raise ValueError(
        "Could not find .assistant/skills in this workspace. "
        f"Tried: {list(seen)}. Set the source_skills_path widget if the repo lives elsewhere."
    )


def list_skill_dirs(source_skills_path: str) -> list[str]:
    if os.path.isdir(source_skills_path):
        return sorted(
            name for name in os.listdir(source_skills_path)
            if os.path.isdir(os.path.join(source_skills_path, name))
        )

    return [
        item.name.rstrip("/")
        for item in dbutils.fs.ls(to_file_uri(source_skills_path))
        if item.isDir()
    ]


def copy_skill_dir(source_skill_path: str, dest_skill_path: str) -> None:
    if os.path.isdir(source_skill_path):
        dbutils.fs.mkdirs(dest_skill_path)
        for root, dirs, files in os.walk(source_skill_path):
            rel_path = os.path.relpath(root, source_skill_path)
            dest_dir = dest_skill_path if rel_path == "." else f"{dest_skill_path}/{rel_path}".replace("\\\\", "/")
            dbutils.fs.mkdirs(dest_dir)
            for file_name in files:
                src_file = os.path.join(root, file_name)
                dst_file = f"{dest_dir}/{file_name}"
                dbutils.fs.cp(f"file:{src_file}", dst_file)
        return

    dbutils.fs.cp(source_skill_path, dest_skill_path, recurse=True)


username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
source_skills_path = resolve_source_skills_path()
dest_skills_path = f"/Users/{username}/.assistant/skills"

dbutils.fs.mkdirs(to_file_uri(f"/Users/{username}/.assistant"))
dbutils.fs.mkdirs(to_file_uri(dest_skills_path))

copied_skills = []
for skill_name in list_skill_dirs(source_skills_path):
    source_skill_path = os.path.join(source_skills_path, skill_name)
    dest_skill_path = f"{to_file_uri(dest_skills_path)}/{skill_name}"
    copy_skill_dir(source_skill_path, dest_skill_path)
    copied_skills.append(skill_name)

print(f"Source: {source_skills_path}")
print(f"Destination: {dest_skills_path}")
print(f"Copied {len(copied_skills)} skills:")
for skill_name in copied_skills:
    print(f"  - {skill_name}")